In [2]:
#
# Cell 0: load libraries
#

import numpy as np
from pathlib import Path
import pickle


In [4]:
#
# Cell 1: path to read input data
#

#directory_path = Path("D:/ECG/XY_diff_class_0_aug_1000") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_800") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_600") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_400") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_aug_200") # chose the right path beforhand
directory_path = Path("D:/ECG/XY_diff_class_0_aug_mixed") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0_mixed") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_0") # chose the right path beforhand


In [6]:
#
# Cell 2: load X-data
#


#loaded_arrays = np.load(directory_path/'X_np_arrays.npz',allow_pickle=True)

#loaded_arrays = np.load(directory_path/'X_np_arrays.npz',allow_pickle=True)
#loaded_arrays = np.load(directory_path/'X_np_arrays_2.npz',allow_pickle=True)
loaded_arrays = np.load(directory_path/'X_np_arrays_3.npz',allow_pickle=True)

# chose the file to restore:
X_train_in = loaded_arrays['a']
X_val_in = loaded_arrays['b']
X_test_in = loaded_arrays['c']

print("X_train_in shape:", X_train_in.shape)
print("X_val_in:", X_val_in.shape)
print("X_test_in:", X_test_in.shape)

loaded_arrays.close()

X_train_in shape: (43135, 1000, 12)
X_val_in: (14378, 1000, 12)
X_test_in: (6378, 1000, 12)


In [8]:
#
# Cell 3: modify X-data: chose sequence length and lead 
# Note: if number of leads >1 is chosen the models has to be adapted accordingly
#

#chose lead:
lead = 1
#chose lenght of sequence (1000, 800, 600, 400, 200)
seq_length = 1000

X_train = X_train_in[:, 0:seq_length,lead].copy()
X_val = X_val_in[:, 0:seq_length,lead].copy()
X_test = X_test_in[:,0:seq_length,lead].copy()

print("X_train shape:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train shape: (43135, 1000)
X_val: (14378, 1000)
X_test: (6378, 1000)


In [10]:
print("len X_train:", len(X_train))
print("len X_val:", len(X_val))
print("len X_test:", len(X_test))

len X_train: 43135
len X_val: 14378
len X_test: 6378


In [12]:
#
# Cell 4: load y-data 
#

#with open(directory_path/'Y_data.pkl', 'rb') as f:
#   Y_final_dfs = pickle.load(f) # deserialize using load()
#f.close()  

#with open(directory_path/'Y_data_2.pkl', 'rb') as f:
#   Y_final_dfs = pickle.load(f) # deserialize using load()
#f.close() 


with open(directory_path/'Y_data_3.pkl', 'rb') as f:
   Y_final_dfs = pickle.load(f) # deserialize using load()
f.close()

#Y_final_dfs

In [14]:
#
# Cell 5: converts the content of df, column 'codes_2', into a list 
#


y_train=list(Y_final_dfs['train_df']['codes_2']=='SR')
y_val=list(Y_final_dfs['val_df']['codes_2']=='SR')
y_test=list(Y_final_dfs['test_df']['codes_2']=='SR')


print("len y_train:", len(y_train))
print("len y_val:", len(y_val))
print("len y_test:", len(y_test))

len y_train: 43135
len y_val: 14378
len y_test: 6378


In [16]:
#
# Cell 6: converts the lists of cell 5 into arrays 
#


from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_val = encoder.fit_transform(y_val)
y_test = encoder.fit_transform(y_test)

print(y_train)

print("len y_train:", len(y_train))
print("len y_val:", len(y_val))
print("len y_test:", len(y_test))

[0 0 1 ... 1 0 1]
len y_train: 43135
len y_val: 14378
len y_test: 6378


In [18]:
#
# Cell 7: calculates class_weight  of classes 
#

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
class_weight

{0: 0.9529227234568992, 1: 1.0519705394595649}

In [20]:
#
# Cell 8: final preparation of the input data for model training 
#

# -----------------------------
# 0) Prepare data shapes
# -----------------------------

def ensure_3d(X):
    X = np.asarray(X)
    if X.ndim == 2:          # (N, 1000) -> (N, 1000, 1)
        X = X[..., None]
    return X

# Use your arrays here:
# X_train, y_train (0/1), X_val, y_val (0/1)
X_train = ensure_3d(X_train).astype(np.float32)
X_val   = ensure_3d(X_val).astype(np.float32)
X_test   = ensure_3d(X_test).astype(np.float32)
y_train = np.asarray(y_train).astype(np.int32)
y_val   = np.asarray(y_val).astype(np.int32)
y_test   = np.asarray(y_test).astype(np.int32)

input_shape = X_train.shape[1:]  # (1000, 1)

class_weight = {0: 0.953, 1: 1.052}


In [22]:
print("X_train shape:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

X_train shape: (43135, 1000, 1)
X_val: (14378, 1000, 1)
X_test: (6378, 1000, 1)
y_train shape: (43135,)
y_val shape: (14378,)
y_test shape: (6378,)


In [24]:

#
# Cell 9: creates a folder for data storage.
#


directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path_input}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path_input}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_pathinput}'.")
except Exception as e:
     print(f"An error occurred: {e}")

Directory 'D:\ECG\input_for_model' already exists.


In [26]:
#
# Cell 10: store data.
#

np.savez_compressed(directory_path_input/'X_y_input.npz', a=X_train, 
                    b=y_train, c=X_val, d=y_val, e=X_test, f=y_test)

#np.savez_compressed(directory_path_input/'X_y_input.npz', a=X_train, 
#                   b=y_train, c=X_val, d=y_val)